# Two-Tower Business Matching Model
Set the four data paths, then run the notebook. It trains a shared Character-CNN encoder and saves the model weights.

In [10]:
from pathlib import Path
import sys
import torch

# Change only these paths after attaching your data in Kaggle.
SOURCE1_PATH = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce1.tsv')
SOURCE2_PATH = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce2.tsv')
SOURCE3_PATH = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce3.tsv')
LABELS_PATH  = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\labeledPairs.tsv')

EPOCHS = 5
BATCH_SIZE = 128
MAX_LABEL_ROWS = 10000  # Example: 200000 for a quick experiment
MODEL_OUTPUT = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\src\checkpoints')

# In Kaggle, upload/attach this src folder or set this path to its location.
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Training device:', DEVICE)


Training device: cpu


In [11]:
import time
import pandas as pd

# Sirf model ke kaam ke columns load karenge (speed badhane ke liye)
COLS = ["entity_id", "name_norm", "name_ascii", "address_norm", "country_norm", "legal_form"]

print("1. Loading Source 1...")
t0 = time.time()
df_s1 = pd.read_csv(SOURCE1_PATH, sep="\t", usecols=COLS, dtype=str).fillna("")
print(f"   Done in {time.time()-t0:.1f}s | Total rows: {len(df_s1):,}")

print("2. Loading Source 2...")
t0 = time.time()
df_s2 = pd.read_csv(SOURCE2_PATH, sep="\t", usecols=COLS, dtype=str).fillna("")
print(f"    Done in {time.time()-t0:.1f}s | Total rows: {len(df_s2):,}")

print("3. Loading Source 3...")
t0 = time.time()
df_s3 = pd.read_csv(SOURCE3_PATH, sep="\t", usecols=COLS, dtype=str).fillna("")
print(f"    Done in {time.time()-t0:.1f}s | Total rows: {len(df_s3):,}")

print("4. Loading Labeled Pairs...")
t0 = time.time()
# MAX_LABEL_ROWS (e.g. 50000 ya 100000)
df_labels = pd.read_csv(LABELS_PATH, sep="\t", nrows=MAX_LABEL_ROWS, dtype=str, keep_default_na=False)
df_labels["label"] = df_labels["label"].astype(float).astype(int)
print(f"    Done in {time.time()-t0:.1f}s | Total pairs: {len(df_labels):,}")


1. Loading Source 1...
   Done in 5.9s | Total rows: 2,206,821
2. Loading Source 2...
    Done in 16.6s | Total rows: 5,034,616
3. Loading Source 3...
    Done in 21.4s | Total rows: 5,285,603
4. Loading Labeled Pairs...
    Done in 0.1s | Total pairs: 10,000


In [12]:
import sys
from pathlib import Path

# Parent folder 'src' add karein
if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

try:
    from model.two_tower import generate_embeddings
    from model.train_utils import train_model
except ModuleNotFoundError:
    from two_tower import generate_embeddings
    from train_utils import train_model

print("Combining records in RAM...")
t0 = time.time()

# 1. Teeno sources ko combine karke index karein
records = pd.concat([df_s1, df_s2, df_s3], ignore_index=True)
records = records.drop_duplicates("entity_id").set_index("entity_id", drop=False)

# 2. Labels filter karein
labels = df_labels[
    df_labels["source1_entity_id"].isin(records.index) & 
    df_labels["target_entity_id"].isin(records.index)
].copy()

print(f" Ready in {time.time()-t0:.1f}s!\n")
print('Loaded business records:', len(records))
print('Positive pairs (1)     :', int((labels.label == 1).sum()))
print('Negative pairs (0)     :', int((labels.label == 0).sum()))


Combining records in RAM...
 Ready in 17.0s!

Loaded business records: 12527040
Positive pairs (1)     : 1694
Negative pairs (0)     : 8306


In [14]:
model, tokenizer, legal_vocab, history = train_model(
    records, labels, MODEL_OUTPUT,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

print('Saved model files:')
for path in sorted(MODEL_OUTPUT.iterdir()):
    print('-', path.name)



[INFO] Starting Training on Device: cpu
[INFO] Train Batches: 11 | Validation Batches: 3
[INFO] Total Epochs: 5 | Batch Size: 128 | Learning Rate: 0.001



Epoch [01/05]  |  Train Loss: 1.8421  |  Val Loss: 0.0939  [BEST MODEL SAVED]


Epoch [02/05]  |  Train Loss: 0.3873  |  Val Loss: 0.0547  [BEST MODEL SAVED]


Epoch [03/05]  |  Train Loss: 0.1558  |  Val Loss: 0.0559  


Epoch [04/05]  |  Train Loss: 0.0745  |  Val Loss: 0.0436  [BEST MODEL SAVED]


Epoch [05/05]  |  Train Loss: 0.0483  |  Val Loss: 0.0459  
Saved model files:
- best.pt
- config.json
- history.json
- last.pt
- legal_form_vocab.json
- tokenizer.json


In [15]:
# Example: generate embeddings for a few records.
sample = records.iloc[:5]
vectors = generate_embeddings(model, tokenizer, legal_vocab, sample, DEVICE)
print('Embedding shape:', vectors.shape)  # expected: (5, 128)
print('First vector norm:', (vectors[0] ** 2).sum() ** 0.5)


Embedding shape: (5, 128)
First vector norm: 0.99999994


In [17]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# 1. Device aur Model Directory check
model_dir = Path(MODEL_OUTPUT)
device = torch.device(DEVICE if 'DEVICE' in globals() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print(f"[INFO] Loading saved weights from: {model_dir.resolve()}")

# 2. Tokenizer aur Legal Form Vocab load karein
from model.two_tower import CharTokenizer, LegalFormVocab, EntityEncoder, generate_embeddings

tokenizer = CharTokenizer.load(model_dir / "tokenizer.json")
legal_vocab = LegalFormVocab.load(model_dir / "legal_form_vocab.json")

# 3. Model Architecture initialize karein aur Weights load karein
model = EntityEncoder(
    char_vocab_size=len(tokenizer.vocab) + 2,
    legal_vocab_size=len(legal_vocab.vocab) + 2
).to(device)

checkpoint = torch.load(model_dir / "best.pt", map_location=device)
model.load_state_dict(checkpoint["model"])
model.eval()

print(f"[INFO] Successfully loaded model from epoch {checkpoint.get('epoch', 'N/A')}")

# 4. Candidates Pool Index karein (Target Source 2/3 records)
target_ids = labels["target_entity_id"].unique()
candidates_df = records.loc[records.index.isin(target_ids)].copy()

print(f"[INFO] Computing embeddings for {len(candidates_df)} candidates...")
candidate_embeddings = generate_embeddings(
    model, tokenizer, legal_vocab, candidates_df, device=device, batch_size=512
)
candidate_ids = candidates_df["entity_id"].to_numpy()
print("[INFO] Candidate index ready.")


# 5. Prediction Function
def predict_matches(query_id: str, top_k: int = 5):
    if query_id not in records.index:
        print(f"[ERROR] ID '{query_id}' not found in records.")
        return None

    query_row = records.loc[[query_id]]
    query_vec = generate_embeddings(model, tokenizer, legal_vocab, query_row, device=device)

    # Cosine Similarity (Dot Product)
    scores = (query_vec @ candidate_embeddings.T).flatten()
    top_indices = np.argsort(-scores)[:top_k]

    # Ground truth positive check
    actual_positives = set(
        labels[(labels["source1_entity_id"] == query_id) & (labels["label"] == 1)]["target_entity_id"]
    )

    q = query_row.iloc[0]
    print("\n" + "=" * 65)
    print(f"QUERY ID  : {query_id}")
    print(f"Name      : {q.get('name_norm', '')}")
    print(f"Address   : {q.get('address_norm', '')}")
    print(f"Country   : {q.get('country_norm', '')}")
    print("=" * 65)
    print(f"TOP {top_k} PREDICTED MATCHES:")
    print("=" * 65)

    for rank, idx in enumerate(top_indices, start=1):
        cand_id = candidate_ids[idx]
        cand_row = candidates_df.iloc[idx]
        score = scores[idx]
        match_flag = "[GROUND TRUTH MATCH]" if cand_id in actual_positives else ""

        print(f"Rank {rank} | Score: {score:.4f} {match_flag}")
        print(f"  ID     : {cand_id}")
        print(f"  Name   : {cand_row.get('name_norm', '')}")
        print(f"  Address: {cand_row.get('address_norm', '')}")
        print(f"  Country: {cand_row.get('country_norm', '')}")
        print("-" * 65)


# 6. Sample Prediction Run Karein
sample_query = labels[labels["label"] == 1].iloc[20]["source1_entity_id"]
predict_matches(sample_query, top_k=5)


[INFO] Loading saved weights from: C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\src\checkpoints
[INFO] Successfully loaded model from epoch 4
[INFO] Computing embeddings for 9994 candidates...
[INFO] Candidate index ready.

QUERY ID  : S1-755362802
Name      : prabhav business center
Address   : 797 lake town block a kolkata howrah west bengal
Country   : india
TOP 5 PREDICTED MATCHES:
Rank 1 | Score: 0.9497 [GROUND TRUTH MATCH]
  ID     : S3-440254853
  Name   : mr prabhav business services
  Address: plot 334 797 lake town block a kolkata howrah পশ্চিমবঙ্গ
  Country: india
-----------------------------------------------------------------
Rank 2 | Score: 0.8871 [GROUND TRUTH MATCH]
  ID     : S2-587230276
  Name   : prabhav center center
  Address: hn 567 797 lake town block a kolkata null west bengal
  Country: india
-----------------------------------------------------------------
Rank 3 | Score: 0.7176 
  ID     : S3-59430361
  Name   : raheja constructions
  A

In [18]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")
print(f"Model Size in RAM    : {(total_params * 4) / (1024 ** 2):.2f} MB")


Total Parameters     : 332,544
Trainable Parameters : 332,544
Model Size in RAM    : 1.27 MB


In [19]:
print(f"{'Layer Name':<45} | {'Shape':<25} | {'Parameters':<12}")
print("-" * 88)

for name, param in model.named_parameters():
    print(f"{name:<45} | {str(tuple(param.shape)):<25} | {param.numel():>10,}")

print("-" * 88)
print(f"{'TOTAL':<45} | {'':<25} | {sum(p.numel() for p in model.parameters()):>10,}")


Layer Name                                    | Shape                     | Parameters  
----------------------------------------------------------------------------------------
name_cnn.embedding.weight                     | (224, 48)                 |     10,752
name_cnn.convs.0.weight                       | (64, 48, 3)               |      9,216
name_cnn.convs.0.bias                         | (64,)                     |         64
name_cnn.convs.1.weight                       | (64, 48, 5)               |     15,360
name_cnn.convs.1.bias                         | (64,)                     |         64
name_cnn.convs.2.weight                       | (64, 48, 7)               |     21,504
name_cnn.convs.2.bias                         | (64,)                     |         64
name_cnn.projection.weight                    | (128, 192)                |     24,576
name_cnn.projection.bias                      | (128,)                    |        128
address_cnn.embedding.weight           